# Kandinsky 6 Lite — Diffusers Generation · T2V / I2V

Практический runner без ComfyUI и учебных разделов. GPU **T4 ×2**, Internet ON. Выберите preset / Smoke или Baseline, для I2V укажите reference из Kaggle Input; выполняйте ячейки по порядку.

Smoke: 25 сгенерированных → 24 экспортированных кадра, 1 s. Baseline: 121 → 120 кадров, 5 s, 24 FPS. Путь подготовлен по официальному API; полный pretrained запуск и two-GPU speedup ещё не измерены.

[Diffusers API](https://huggingface.co/docs/diffusers/en/api/pipelines/kandinsky6) · [Lite checkpoint](https://huggingface.co/kandinskylab/Kandinsky-6.0-Lite-distill-5s-Diffusers) · [README и presets](https://github.com/vladislav-vasilenko/generative-ai-research/tree/main/kandinsky-6/generation) · [Сохранённый вариант ComfyUI](https://www.kaggle.com/code/applehawk/15-kaggle-t4-lite-distill).

Учебные материалы отдельно: [MIT 6.S184](https://diffusion.csail.mit.edu/2026/index.html) · [конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [оглавление проекта](https://github.com/vladislav-vasilenko/generative-ai-research/blob/main/curriculum/CONTENTS.md).


## 1. Среда

В редакторе Kaggle включите **Internet → On**. Если `pypi.org` не разрешается, проверьте переключатель; при уже включённом Internet возможен временный DNS-сбой. Ячейка проверяет доступ к PyPI до pip и пропускает установку, когда версии уже подходят. Pillow не понижается: Pillow 13 из образа Kaggle подходит, поэтому на свежей сессии перезапуск ядра не нужен и «Save & Run All» проходит. pip может предупредить, что неиспользуемый здесь Gradio требует `pillow<13`; на запуск это не влияет. Если всё же изменилась уже импортированная библиотека, потребуется Restart Python kernel, затем повтор с первой ячейки.


In [ ]:
import os, sys, subprocess, socket
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError
from packaging.requirements import Requirement
import torch

if not Path('/kaggle/working').is_dir() or torch.cuda.device_count() < 2:
    raise RuntimeError('Run in Kaggle with GPU T4 ×2. No installation or model downloads started.')
for i in range(2):
    print(i, torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory/2**30,2),'GiB')
print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda)
SCRATCH=Path('/kaggle/temp/k6-diffusers')
RESULTS=Path('/kaggle/working/k6-diffusers-results')
SCRATCH.mkdir(parents=True,exist_ok=True);RESULTS.mkdir(parents=True,exist_ok=True)
os.environ.update(HF_HUB_DISABLE_TELEMETRY='1',HF_HUB_DISABLE_XET='1',TOKENIZERS_PARALLELISM='false')
pins={'diffusers':'0.42.1','transformers':'5.22.1','accelerate':'1.13.0'}


def pending_dependencies(requirements, version_lookup=version):
    missing = []
    for item in requirements:
        requirement = Requirement(item)
        try:
            installed = version_lookup(requirement.name)
        except PackageNotFoundError:
            missing.append(item)
            continue
        if installed not in requirement.specifier:
            missing.append(item)
    return missing


def check_pypi_access():
    url = 'https://pypi.org/simple/diffusers/'
    try:
        with urlopen(Request(url, method='HEAD'), timeout=10):
            pass
    except HTTPError as error:
        raise RuntimeError(f'PyPI responds with HTTP {error.code}. Check Kaggle Internet access; pip has not started.') from error
    except URLError as error:
        cause = ('DNS cannot resolve pypi.org' if isinstance(error.reason, socket.gaierror)
                 else f'HTTPS connection failed: {error.reason}')
        raise RuntimeError(cause + '. In Kaggle editor set Internet to On, then retry this cell. '
                           'If Internet is already On, retry after the temporary network outage. pip has not started.') from error
    except (TimeoutError, OSError) as error:
        raise RuntimeError('PyPI network probe timed out/failed. Check Kaggle Internet and retry this cell. pip has not started.') from error


def check_loaded_dependency_versions():
    # A pip upgrade cannot replace modules already imported by this Python kernel.
    for distribution, module_name in [('pillow','PIL'),('diffusers','diffusers'),
                                      ('transformers','transformers'),('accelerate','accelerate')]:
        module = sys.modules.get(module_name)
        active = getattr(module, '__version__', None)
        if active is not None and str(active) != version(distribution):
            raise RuntimeError(f'{distribution}: kernel has {active}, installed files are {version(distribution)}. '
                               'Restart the Python kernel, then run this notebook again from cell 1.')


def ensure_runtime_dependencies(requirements, constraint_file):
    pending = pending_dependencies(requirements)
    if pending:
        print('Dependencies to install/update:', pending, flush=True)
        check_pypi_access()
        subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
            '--retries','1','--timeout','15','--upgrade-strategy','only-if-needed',
            '-c',str(constraint_file),*requirements],check=True)
    else:
        print('Compatible dependencies already installed; skipping installation and PyPI probe.',flush=True)
    check_loaded_dependency_versions()
    return bool(pending)


requirements = [name+'=='+v for name,v in pins.items()] + [
    'av>=18.0.0,<19', 'safetensors>=0.8.0', 'pillow>=10.4']
constraints=[]
for name in ['torch','torchvision','torchaudio']:
    try:constraints.append(name+'=='+version(name))
    except PackageNotFoundError:pass
constraint_file=SCRATCH/'torch-constraints.txt'
constraint_file.write_text('\n'.join(constraints)+'\n')
ensure_runtime_dependencies(requirements,constraint_file)
(RESULTS/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','--disable-pip-version-check','freeze'],text=True))


## 2. Что генерировать

`orbit_woman`, `strange_tram`, `mechanical_bloom`, `room_assembly` — T2V. `mario_poster`, `mario_pipe_illustration`, `mario_kandinsky_growth` — I2V для приложенных reference. Сами картинки не опубликованы: загрузите их в Input и вставьте путь. `LOCAL_MODEL_DIR` принимает только полный **Diffusers layout**, не ComfyUI single-file checkpoint.

Для `mario_kandinsky_growth` сначала подготовьте один PNG/JPEG с логотипом уже на грибе: в metadata пресета есть отдельный prompt подготовки кадра. SVG не подаётся вторым reference. Точная песня берётся из вашего `MUSIC_PATH` при экспорте; модель генерирует оригинальный funk/new-wave фон.

Таймлайн — текстовое пожелание, не точные keyframes. Distill использует CFG=1: negative prompt не создаёт отдельную guidance-ветвь; существенные ограничения записаны и в положительном prompt.


In [ ]:
PRESET_NAME='orbit_woman'
PROFILE='smoke'                 # 'baseline' for 5 s
SEED=42
IMAGE_PATH=None                # e.g. '/kaggle/input/my-references/images-4.jpeg'
ASPECT='auto'                  # auto / landscape / portrait / square
LOCAL_MODEL_DIR=None           # optional '/kaggle/input/.../full-diffusers-snapshot'
AUDIO_MODE='generated'          # 'music' or 'mix' to use your own audio file
MUSIC_PATH=None                 # e.g. '/kaggle/input/my-audio/burning_down_the_house.aac'
MUSIC_OFFSET_SECONDS=0.0
MUSIC_GAIN=0.65
DIT_PLACEMENT='balanced'        # 'balanced' (layer split over 2 GPUs), 'cuda:0' or 'cuda:1' (whole DiT on one T4)
MEASURE_PLACEMENTS=[]           # optional extra denoise-only runs, e.g. [('baseline','balanced'),('baseline','cuda:0')]
MODEL_ID='kandinskylab/Kandinsky-6.0-Lite-distill-5s-Diffusers'
MODEL_REVISION='4c487e8b4aeeb9059e5a77905aa2ddb59ec28f81'


### Presets: редактируемые сценарии


In [ ]:
PRESETS = {'orbit_woman': {'metadata': {'prompt_name': 'Woman / continuous camera orbit',
                              'mode': 't2v',
                              'base_style': 'Cinematic live-action fashion film, natural skin texture, '
                                            'restrained color grading, realistic cloth physics.'},
                 'scene_setup': {'room_description': 'An adult woman, about 28 years old, stands in a '
                                                     'circular stone courtyard. She wears a deep emerald '
                                                     'coat and a cream silk scarf. Archways and foreground '
                                                     'columns establish clear depth.',
                                 'camera_setup': 'One uninterrupted medium shot at eye level; a smooth '
                                                 'camera orbit, no cuts, constant camera-to-subject '
                                                 'distance.',
                                 'lighting': 'Warm late-afternoon side light; stable exposure and soft '
                                             'moving highlights.'},
                 'elements': {'initial_state': ['The woman faces the camera, both feet planted; hair tied '
                                                'loosely; scarf rests on her right shoulder.']},
                 'reference_filename_hint': None,
                 'profiles': {'smoke': {'camera_setup': 'The camera makes a small, smooth 15-degree '
                                                        'clockwise arc; keep her face centered. Do not '
                                                        'attempt a full circle in one second.',
                                        'timeline': [{'start': 0,
                                                      'end': 0.35,
                                                      'action': 'Start from a frontal three-quarter view. '
                                                                'The camera begins its arc, a near column '
                                                                'slides slowly across the background.',
                                                      'audio_cue': 'Quiet courtyard air and a soft breeze.'},
                                                     {'start': 0.35,
                                                      'end': 1,
                                                      'action': 'Continue the arc smoothly. She turns her '
                                                                'eyes toward the lens and the scarf lifts '
                                                                'slightly, then begins to settle. Same face '
                                                                'and outfit throughout.',
                                                      'audio_cue': 'A brief silk rustle; no dialogue or '
                                                                   'music.'}]},
                              'baseline': {'camera_setup': 'The camera performs a smooth approximately '
                                                           '160-degree clockwise orbit from a front '
                                                           'three-quarter view to a rear three-quarter view. '
                                                           'Keep the woman centered at the same scale; '
                                                           'archways pass with strong parallax.',
                                           'timeline': [{'start': 0,
                                                         'end': 1,
                                                         'action': 'Establish her face and emerald coat. The '
                                                                   'camera starts moving; a foreground '
                                                                   'column passes along the edge.',
                                                         'audio_cue': 'Soft courtyard ambience.'},
                                                        {'start': 1,
                                                         'end': 3.6,
                                                         'action': 'The camera continues around her right '
                                                                   'side. She slowly turns her head to '
                                                                   'follow the lens, while a single gust '
                                                                   'pulls her scarf outward. Her body '
                                                                   'remains planted.',
                                                         'audio_cue': 'One gentle gust, silk fluttering, '
                                                                      'distant birds.'},
                                                        {'start': 3.6,
                                                         'end': 5,
                                                         'action': 'Arrive at the rear three-quarter view. '
                                                                   'The scarf settles onto her shoulder and '
                                                                   'the camera eases to a stop; preserve her '
                                                                   'silhouette and clothing.',
                                                         'audio_cue': 'The breeze fades; no speech or '
                                                                      'soundtrack.'}]}},
                 'negative_prompts': ['cuts',
                                      'flicker',
                                      'identity drift',
                                      'extra limbs',
                                      'warped hands',
                                      'text overlays',
                                      'watermarks']},
 'strange_tram': {'metadata': {'prompt_name': 'Retrofuturist tram / practical creature effects',
                               'mode': 't2v',
                               'base_style': 'Surreal retrofuturist dark comedy, 1970s science-fiction '
                                             'cinema, tactile practical creature effects, worn brass, '
                                             'painted miniatures, muted teal and amber, subtle film grain.'},
                  'scene_setup': {'room_description': 'A rain-wet tram stop in an imaginary industrial city. '
                                                      'An original pear-shaped, blue-skinned commuter with '
                                                      'tiny spectacles and a tan raincoat waits beside a '
                                                      'brass ticket machine. He has two arms and two legs.',
                                  'camera_setup': 'One continuous eye-level tracking shot; restrained '
                                                  'handheld texture without sudden shaking.',
                                  'lighting': 'Overcast daylight mixed with warm tungsten light in the '
                                              'ticket machine.'},
                  'elements': {'initial_state': ['The commuter holds one paper ticket. The ticket slot is '
                                                 'shut. Steam drifts behind the stop.']},
                  'reference_filename_hint': None,
                  'profiles': {'smoke': {'camera_setup': 'A short, slow push-in, without changing the view '
                                                         'angle.',
                                         'timeline': [{'start': 0,
                                                       'end': 0.4,
                                                       'action': 'The brass ticket slot clicks open. The '
                                                                 'commuter freezes and raises one eyebrow.',
                                                       'audio_cue': 'One dry metallic click over quiet '
                                                                    'rain.'},
                                                      {'start': 0.4,
                                                       'end': 1,
                                                       'action': 'A small mechanical tongue briefly taps the '
                                                                 'edge of his ticket, then retracts. The '
                                                                 'commuter blinks in deadpan disbelief.',
                                                       'audio_cue': 'A soft rubbery squeak and a tiny bell; '
                                                                    'no speech.'}]},
                               'baseline': {'camera_setup': 'Track slowly left by about one metre as the '
                                                            'action unfolds; preserve the same subject and '
                                                            'scale.',
                                            'timeline': [{'start': 0,
                                                          'end': 1,
                                                          'action': 'The commuter inserts the ticket halfway '
                                                                    'into the slot.',
                                                          'audio_cue': 'Rain, faint tram rails humming.'},
                                                         {'start': 1,
                                                          'end': 2.8,
                                                          'action': 'A small mechanical tongue takes the '
                                                                    'ticket, curls it into a neat tube, and '
                                                                    'pushes it back out. The creature '
                                                                    'watches without moving his feet.',
                                                          'audio_cue': 'A wet rubber squeak followed by '
                                                                       'delicate brass clicks.'},
                                                         {'start': 2.8,
                                                          'end': 4.2,
                                                          'action': 'The commuter cautiously takes the '
                                                                    'rolled ticket with the same hand and '
                                                                    'tilts his head. A tram headlamp slides '
                                                                    'into the far background.',
                                                          'audio_cue': 'A distant tram approaches and rings '
                                                                       'once.'},
                                                         {'start': 4.2,
                                                          'end': 5,
                                                          'action': 'Hold his puzzled expression as the slot '
                                                                    'closes with exaggerated politeness. '
                                                                    'Maintain tactile material detail and a '
                                                                    'coherent street.',
                                                          'audio_cue': 'One final click; quiet rain, no '
                                                                       'narration.'}]}},
                  'negative_prompts': ['cuts',
                                       'flicker',
                                       'identity drift',
                                       'extra limbs',
                                       'warped hands',
                                       'text overlays',
                                       'watermarks']},
 'mechanical_bloom': {'metadata': {'prompt_name': 'Biomechanical greenhouse / unfolding flower',
                                   'mode': 't2v',
                                   'base_style': 'Tactile miniature science-fiction film, '
                                                 'stop-motion-inspired creature design with smooth motion, '
                                                 'copper mechanisms and translucent botanical tissue, '
                                                 'detailed macro photography.'},
                      'scene_setup': {'room_description': 'An original small flower made of copper ribs and '
                                                          'translucent pale-blue petals grows from a cracked '
                                                          'ceramic pot on a workbench. The background is an '
                                                          'unfocused greenhouse.',
                                      'camera_setup': 'One continuous macro shot, shallow depth of field; '
                                                      'focus remains on the flower head.',
                                      'lighting': 'Soft green backlight through glass with a warm practical '
                                                  'lamp on the workbench.'},
                      'elements': {'initial_state': ['One closed flower bud; a drop of water hangs from a '
                                                     'copper leaf.']},
                      'reference_filename_hint': None,
                      'profiles': {'smoke': {'camera_setup': 'A tiny sideways camera slide creating gentle '
                                                             'foreground parallax.',
                                             'timeline': [{'start': 0,
                                                           'end': 0.5,
                                                           'action': 'The bud quivers and the first pair of '
                                                                     'petals separates.',
                                                           'audio_cue': 'A quiet metallic tick.'},
                                                          {'start': 0.5,
                                                           'end': 1,
                                                           'action': 'The petals open a little farther and '
                                                                     'release one small puff of glowing '
                                                                     'pollen. The pot and bench stay fixed.',
                                                           'audio_cue': 'A soft pneumatic puff and a '
                                                                        'delicate chime; no voice.'}]},
                                   'baseline': {'camera_setup': 'A gentle 45-degree orbit around the flower '
                                                                'at a fixed macro distance.',
                                                'timeline': [{'start': 0,
                                                              'end': 1,
                                                              'action': 'The closed bud slowly tilts toward '
                                                                        'the warm lamp.',
                                                              'audio_cue': 'A faint mechanism starts '
                                                                           'ticking.'},
                                                             {'start': 1,
                                                              'end': 3.4,
                                                              'action': 'Six translucent petals unfold in a '
                                                                        'clear sequence as the copper ribs '
                                                                        'extend. One drop of water falls '
                                                                        'from the leaf.',
                                                              'audio_cue': 'Delicate ratchet clicks, one '
                                                                           'water drop.'},
                                                             {'start': 3.4,
                                                              'end': 4.4,
                                                              'action': 'The flower opens fully, releasing a '
                                                                        'single drifting cloud of luminous '
                                                                        'pollen.',
                                                              'audio_cue': 'A soft breath-like puff.'},
                                                             {'start': 4.4,
                                                              'end': 5,
                                                              'action': 'The flower settles and the camera '
                                                                        'eases to a stop. Keep the same pot, '
                                                                        'leaf and bench.',
                                                              'audio_cue': 'The ticking subsides into '
                                                                           'greenhouse ambience.'}]}},
                      'negative_prompts': ['cuts',
                                           'flicker',
                                           'identity drift',
                                           'extra limbs',
                                           'warped hands',
                                           'text overlays',
                                           'watermarks']},
 'room_assembly': {'metadata': {'prompt_name': 'Scandinavian room / one bed assembly',
                                'mode': 't2v',
                                'base_style': 'Photorealistic Scandinavian furniture commercial, clean '
                                              'geometric composition, natural materials, restrained '
                                              'palette.'},
                   'scene_setup': {'room_description': 'An empty sunlit room with white walls, a pale oak '
                                                       'floor and one large window. The room is unoccupied.',
                                   'camera_setup': 'A single fixed wide shot for the entire clip, no camera '
                                                   'movement, no cuts.',
                                   'lighting': 'Soft stable morning light from the left window.'},
                   'elements': {'initial_state': ['One unbranded flat-pack cardboard box in the center of '
                                                  'the room. No other furniture.']},
                   'reference_filename_hint': None,
                   'profiles': {'smoke': {'camera_setup': 'Completely locked-off wide shot.',
                                          'timeline': [{'start': 0,
                                                        'end': 0.3,
                                                        'action': 'The sealed cardboard box begins to '
                                                                  'tremble gently.',
                                                        'audio_cue': 'A low brief cardboard rattle.'},
                                                       {'start': 0.3,
                                                        'end': 1,
                                                        'action': 'Its top flaps pop open and the first pair '
                                                                  'of wooden bed rails rises a short '
                                                                  'distance above the box. The rest stays '
                                                                  'packed.',
                                                        'audio_cue': 'A soft pop and two wooden clacks. The '
                                                                     'room is otherwise quiet.'}]},
                                'baseline': {'camera_setup': 'Completely static wide shot. Keep the window, '
                                                             'walls and floor identical.',
                                             'timeline': [{'start': 0,
                                                           'end': 0.6,
                                                           'action': 'The box trembles and its top flaps '
                                                                     'open.',
                                                           'audio_cue': 'Cardboard rustle and a small pop.'},
                                                          {'start': 0.6,
                                                           'end': 2.5,
                                                           'action': 'Four wooden rails and a few slats lift '
                                                                     'out, rotate into alignment and connect '
                                                                     'into one low minimalist bed frame.',
                                                           'audio_cue': 'A short ordered sequence of wooden '
                                                                        'clicks.'},
                                                          {'start': 2.5,
                                                           'end': 4,
                                                           'action': 'One mattress lowers onto the frame. A '
                                                                     'white duvet unrolls across it; a '
                                                                     'single mustard-yellow throw lands at '
                                                                     'the foot. No other furniture appears.',
                                                           'audio_cue': 'A cushioned thump and soft fabric '
                                                                        'rustling.'},
                                                          {'start': 4,
                                                           'end': 5,
                                                           'action': 'Hold the completed bed in the '
                                                                     'otherwise uncluttered room. All pieces '
                                                                     'settle.',
                                                           'audio_cue': 'A gentle warm ambient chord, no '
                                                                        'speech.'}]}},
                   'negative_prompts': ['cuts',
                                        'flicker',
                                        'identity drift',
                                        'extra limbs',
                                        'warped hands',
                                        'text overlays',
                                        'watermarks']},
 'mario_poster': {'metadata': {'prompt_name': 'Reference 1 / 3D poster character',
                               'mode': 'i2v',
                               'base_style': 'Preserve the reference: polished 3D cartoon rendering, rounded '
                                             'proportions, bright red clothing, crisp poster graphics.'},
                  'scene_setup': {'room_description': 'Use the supplied poster as the first frame and visual '
                                                      'ground truth. Preserve the orange left half, blue '
                                                      'right half, radial rays, two small upper stickers and '
                                                      'the exact character design.',
                                  'camera_setup': 'Locked frontal view; preserve the full composition and '
                                                  'all character proportions.',
                                  'lighting': 'Keep the original soft studio lighting and saturated '
                                              'orange-blue palette.'},
                  'elements': {'initial_state': ['The stocky Mario-like cartoon character wears red '
                                                 'overalls, a red cap with its existing emblem, white gloves '
                                                 'and brown shoes. Keep his identity and body shape.']},
                  'reference_filename_hint': 'images-4.jpeg',
                  'profiles': {'smoke': {'camera_setup': 'Camera completely static; the two upper stickers '
                                                         'and radial background remain motionless.',
                                         'timeline': [{'start': 0,
                                                       'end': 0.4,
                                                       'action': 'The main character blinks once and raises '
                                                                 'his eyebrows slightly.',
                                                       'audio_cue': 'A tiny friendly cartoon exhale.'},
                                                      {'start': 0.4,
                                                       'end': 1,
                                                       'action': 'He makes a small cheerful nod; his cap and '
                                                                 'overall straps follow with a subtle '
                                                                 'secondary bounce, then settle. Keep both '
                                                                 'hands and shoes in their original '
                                                                 'positions.',
                                                       'audio_cue': 'A soft fabric creak and a light comic '
                                                                    'pluck. No spoken words.'}]},
                               'baseline': {'camera_setup': 'A very slow approximately 3-percent push-in, no '
                                                            'orbit; both upper stickers remain fixed graphic '
                                                            'elements.',
                                            'timeline': [{'start': 0,
                                                          'end': 1,
                                                          'action': 'He notices the viewer and blinks, '
                                                                    'keeping the original body shape and '
                                                                    'stance.',
                                                          'audio_cue': 'A short curious cartoon hum.'},
                                                         {'start': 1,
                                                          'end': 2.7,
                                                          'action': 'He lifts his right gloved hand, on the '
                                                                    'left side of the image, to the brim of '
                                                                    'his cap and tips it once. His other '
                                                                    'hand remains beside his body.',
                                                          'audio_cue': 'A gentle cloth rustle and one '
                                                                       'cheerful comic pluck.'},
                                                         {'start': 2.7,
                                                          'end': 4.2,
                                                          'action': 'He lowers that hand and gives a small '
                                                                    'proud shoulder bounce. The red overalls '
                                                                    'react with subtle cloth motion; his '
                                                                    'shoes stay in place.',
                                                          'audio_cue': 'A quiet breathy chuckle without '
                                                                       'intelligible speech.'},
                                                         {'start': 4.2,
                                                          'end': 5,
                                                          'action': 'Return to a relaxed near-original pose. '
                                                                    'The background, stickers and existing '
                                                                    'emblems stay unchanged, with no new '
                                                                    'lettering.',
                                                          'audio_cue': 'A light settling sound and then '
                                                                       'quiet.'}]}},
                  'negative_prompts': ['cuts',
                                       'flicker',
                                       'identity drift',
                                       'extra limbs',
                                       'warped hands',
                                       'text overlays',
                                       'watermarks']},
 'mario_pipe_illustration': {'metadata': {'prompt_name': 'Reference 2 / drawn character on pipe',
                                          'mode': 'i2v',
                                          'base_style': 'Preserve the reference: warm paper background, '
                                                        'hand-drawn ink outlines, muted colors, stippled '
                                                        'shading and limited 2D cartoon animation. Keep it '
                                                        'illustrated throughout.'},
                             'scene_setup': {'room_description': 'Use the supplied square drawing as the '
                                                                 'first frame. The heavy cartoon plumber '
                                                                 'sits on the same green pipe, holding the '
                                                                 'same red-and-white mushroom and yellow '
                                                                 'flower.',
                                             'camera_setup': 'Fixed frontal three-quarter view, no camera '
                                                             'orbit, no change of framing.',
                                             'lighting': 'Keep the original flat warm illustration lighting '
                                                         'and textured cream paper.'},
                             'elements': {'initial_state': ['Preserve the red cap, blue overalls, white '
                                                            'gloves, moustache, heavy body proportions, '
                                                            'mushroom, flower, pipe and floor cracks.']},
                             'reference_filename_hint': '3ad019e7defdecd45abf8be97e287972.jpg',
                             'profiles': {'smoke': {'camera_setup': 'Locked camera; paper texture and floor '
                                                                    'cracks remain stationary.',
                                                    'timeline': [{'start': 0,
                                                                  'end': 0.4,
                                                                  'action': 'The character blinks and looks '
                                                                            'toward the mushroom in his '
                                                                            'raised hand.',
                                                                  'audio_cue': 'A tiny amused nasal hum.'},
                                                                 {'start': 0.4,
                                                                  'end': 1,
                                                                  'action': 'The yellow flower nods gently '
                                                                            'once while his shoulders lift '
                                                                            'with a small silent chuckle. '
                                                                            'Preserve the ink outline and '
                                                                            'original objects.',
                                                                  'audio_cue': 'A soft cloth creak and a '
                                                                               'delicate spring sound; no '
                                                                               'speech.'}]},
                                          'baseline': {'camera_setup': 'Camera completely fixed; preserve '
                                                                       'the square paper illustration and '
                                                                       'keep all props visible.',
                                                       'timeline': [{'start': 0,
                                                                     'end': 1,
                                                                     'action': 'He glances from the mushroom '
                                                                               'to the yellow flower and '
                                                                               'lifts one eyebrow.',
                                                                     'audio_cue': 'A low curious cartoon '
                                                                                  'hum.'},
                                                                    {'start': 1,
                                                                     'end': 2.7,
                                                                     'action': 'His raised gloved hand tilts '
                                                                               'the mushroom toward his face '
                                                                               'without eating it; the '
                                                                               'flower bends playfully '
                                                                               'toward the mushroom. Keep '
                                                                               'the props in their original '
                                                                               'hands.',
                                                                     'audio_cue': 'A gentle springy boing '
                                                                                  'and glove rustle.'},
                                                                    {'start': 2.7,
                                                                     'end': 4.1,
                                                                     'action': 'He leans back slightly and '
                                                                               'chuckles; his shoulders and '
                                                                               'belly move together with one '
                                                                               'restrained '
                                                                               'squash-and-stretch bounce. '
                                                                               'The green pipe remains '
                                                                               'intact.',
                                                                     'audio_cue': 'A short warm comic '
                                                                                  'chuckle; no words or '
                                                                                  'music.'},
                                                                    {'start': 4.1,
                                                                     'end': 5,
                                                                     'action': 'He settles toward the '
                                                                               'original seated pose and the '
                                                                               'flower makes one small final '
                                                                               'nod. Preserve the hand-drawn '
                                                                               'style and paper texture.',
                                                                     'audio_cue': 'A soft settling creak, '
                                                                                  'then quiet.'}]}},
                             'negative_prompts': ['cuts',
                                                  'flicker',
                                                  'identity drift',
                                                  'extra limbs',
                                                  'warped hands',
                                                  'text overlays',
                                                  'watermarks']},
 'mario_kandinsky_growth': {'metadata': {'prompt_name': 'Mario / Kandinsky mushroom / growth gag',
                                         'mode': 'i2v',
                                         'base_style': 'Playful hand-drawn cartoon animation; preserve the '
                                                       'supplied reference style, ink outlines, face, '
                                                       'moustache and costume colors.',
                                         'music_reference': 'Talking Heads - Burning Down the House '
                                                            '(Official Video)',
                                         'reference_preparation': 'Prepare a single PNG/JPEG first frame: '
                                                                  'same character, one large red mushroom '
                                                                  'already in his right glove, supplied '
                                                                  'Kandinsky SVG wordmark composited as a '
                                                                  'crisp mint-and-white decal on the cap. Do '
                                                                  'not pass the SVG as a second conditioning '
                                                                  'image.',
                                         'reference_preparation_prompt': 'Create a clean medium-wide first '
                                                                         'frame from the supplied Mario '
                                                                         'reference. Keep the same character '
                                                                         'identity, red cap, overalls, white '
                                                                         'gloves and warm illustrated '
                                                                         'finish. He holds one large red '
                                                                         'mushroom toward the viewer with '
                                                                         'the cap facing the camera. Place '
                                                                         'the exact supplied Kandinsky '
                                                                         'wordmark on the mushroom cap as a '
                                                                         'composited decal; do not redraw or '
                                                                         'invent the lettering. Leave '
                                                                         'generous empty space above his '
                                                                         'head and around his body for later '
                                                                         'growth. Keep the other hand empty '
                                                                         'and the background simple.'},
                            'reference_filename_hint': 'mario_kandinsky_mushroom_prepared.png',
                            'scene_setup': {'room_description': 'Use the prepared reference as the first '
                                                                'frame. One cartoon plumber holds one red '
                                                                'mushroom whose cap already carries the '
                                                                'supplied Kandinsky logo. Preserve the '
                                                                'reference appearance and background.',
                                            'camera_setup': 'One uninterrupted shot. The camera smoothly '
                                                            'dollies backward as the character grows, '
                                                            'keeping his entire body visible.',
                                            'lighting': 'Stable warm cartoon lighting, unchanged colors and '
                                                        'exposure.'},
                            'elements': {'initial_state': ['He holds the mushroom close to his mouth, its '
                                                           'existing Kandinsky decal facing the camera. His '
                                                           'free hand rests at his side. Plenty of '
                                                           'headroom.']},
                            'profiles': {'smoke': {'camera_setup': 'Keep the camera nearly fixed; a tiny '
                                                                   'backward drift compensates for the '
                                                                   'beginning of growth.',
                                                   'timeline': [{'start': 0,
                                                                 'end': 0.4,
                                                                 'action': 'The character takes one clear '
                                                                           'bite from the edge of the '
                                                                           'mushroom. Preserve the existing '
                                                                           'logo on the remaining visible '
                                                                           'cap; the glove stays attached to '
                                                                           'the same hand.',
                                                                 'audio_cue': 'One crisp cartoon crunch on '
                                                                              'an energetic drum accent.'},
                                                                {'start': 0.4,
                                                                 'end': 1,
                                                                 'action': 'A warm mint-colored pulse passes '
                                                                           'through him. His shoulders rise '
                                                                           'and his belly rounds slightly: '
                                                                           'only the start of the power-up. '
                                                                           'His face and costume stay '
                                                                           'recognizable.',
                                                                 'audio_cue': 'A short rising power-up tone '
                                                                              'over an original syncopated '
                                                                              'new-wave funk instrumental, '
                                                                              'dry drums, elastic bass and '
                                                                              'scratchy rhythm guitar; no '
                                                                              'vocals or quoted melody.'}]},
                                         'baseline': {'camera_setup': 'Begin in a medium-wide view. After '
                                                                      'the bite, dolly smoothly backward and '
                                                                      'tilt upward gently so the growing '
                                                                      'character remains fully inside the '
                                                                      'frame. No cuts.',
                                                      'timeline': [{'start': 0,
                                                                    'end': 1,
                                                                    'action': 'He looks at the viewer, lifts '
                                                                              'the mushroom slightly and '
                                                                              'presents its existing '
                                                                              'Kandinsky decal toward the '
                                                                              'camera. Keep the lettering '
                                                                              'already in the reference.',
                                                                    'audio_cue': 'An original energetic '
                                                                                 'early-1980s-style new-wave '
                                                                                 'funk groove: dry punchy '
                                                                                 'drums, elastic bass, '
                                                                                 'clipped rhythm guitar and '
                                                                                 'short bright synth '
                                                                                 'accents; no vocals or '
                                                                                 'familiar melody.'},
                                                                   {'start': 1,
                                                                    'end': 1.8,
                                                                    'action': 'He takes one bite. A small '
                                                                              'piece is missing from the '
                                                                              'cap; his jaw closes naturally '
                                                                              'and he lowers the same hand.',
                                                                    'audio_cue': 'A satisfying crunchy bite '
                                                                                 'aligned to the beat.'},
                                                                   {'start': 1.8,
                                                                    'end': 3.8,
                                                                    'action': 'One mint-colored pulse '
                                                                              'triggers a continuous cartoon '
                                                                              'power-up. He grows roughly '
                                                                              'one-and-a-half times taller '
                                                                              'and becomes noticeably wider '
                                                                              'and rounder, with fuller '
                                                                              'cheeks, broader shoulders and '
                                                                              'a larger belly. His overalls '
                                                                              'stretch with him; preserve '
                                                                              'his face, moustache, cap, two '
                                                                              'arms and two legs. The camera '
                                                                              'pulls back steadily.',
                                                                    'audio_cue': 'A rising playful synth '
                                                                                 'glissando and soft elastic '
                                                                                 'fabric sounds over the '
                                                                                 'steady funk groove.'},
                                                                   {'start': 3.8,
                                                                    'end': 5,
                                                                    'action': 'Now oversized and comically '
                                                                              'plump, he plants his feet, '
                                                                              'gives one satisfied belly '
                                                                              'bounce and grins. Ease the '
                                                                              'camera to a stop. Keep the '
                                                                              'final larger body stable, '
                                                                              'with no extra objects or '
                                                                              'limbs.',
                                                                    'audio_cue': 'One low comic landing '
                                                                                 'thump, then a confident '
                                                                                 'instrumental accent. No '
                                                                                 'spoken dialogue.'}]}},
                            'negative_prompts': ['cuts',
                                                 'identity change',
                                                 'horror',
                                                 'body disintegration',
                                                 'extra limbs',
                                                 'detached gloves',
                                                 'new text',
                                                 'logo morphing',
                                                 'watermark']}}


## 3. Функции запуска

Обычные Python-ячейки. Qwen/CLIP кодируют оба сценария и выгружаются. По умолчанию (`DIT_PLACEMENT='balanced'`) DiT размещается блоками на двух GPU: это распределение памяти с передачей активаций, GPU работают поочерёдно, двукратного ускорения нет. `'cuda:0'`/`'cuda:1'` держит весь DiT на одной T4 для сравнения. Полный CPU-offload pipeline поверх этого device map не включается. Codecs используют FP32 и штатный Diffusers VAE tiling.


In [ ]:
import contextlib
import gc
import hashlib
import json
import resource
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import torch
from PIL import Image, ImageOps
from safetensors import safe_open
from safetensors.torch import load_file, save_file
from diffusers import (
    AutoencoderKLHunyuanVideo, Kandinsky6TI2VAPipeline,
    Kandinsky6Transformer3DModel, MMAudioVAE, MMAudioVocoder, PiflowScheduler,
)
from diffusers.utils import encode_video
from diffusers.video_processor import VideoProcessor
from transformers import CLIPTextModel, CLIPTokenizer, Qwen2_5_VLForConditionalGeneration, Qwen2_5_VLProcessor


In [ ]:
PROFILES = {
    'smoke': {'num_frames': 25, 'export_frames': 24, 'seconds': 1},
    'baseline': {'num_frames': 121, 'export_frames': 120, 'seconds': 5},
}
SIZES = {'landscape': {'smoke': (432, 256), 'baseline': (864, 480)},
         'portrait': {'smoke': (256, 432), 'baseline': (480, 864)},
         'square': {'smoke': (320, 320), 'baseline': (640, 640)}}


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def compile_prompt(preset, profile):
    spec = preset['profiles'][profile]
    duration = PROFILES[profile]['seconds']
    previous = 0
    visual, sounds = [], []
    for event in spec['timeline']:
        start, end = event['start'], event['end']
        if start != previous or not start < end <= duration:
            raise ValueError(f'Invalid {profile} timeline: {event}')
        previous = end
        visual.append(f"From {start:g} to {end:g} seconds: {event['action']}")
        sounds.append(event['audio_cue'])
    if previous != duration:
        raise ValueError('Timeline must cover the selected export duration')
    scene = preset['scene_setup']
    prompt = ' '.join([preset['metadata']['base_style'], scene['room_description'],
        scene['lighting'], spec['camera_setup'],
        'Initial state: ' + ' '.join(preset['elements']['initial_state']),
        f'One uninterrupted shot, about {duration} seconds.', *visual,
        '<AUDCAP>' + ' '.join(sounds) + '<ENDAUDCAP>'])
    return prompt, ', '.join(preset['negative_prompts'])


def reference_and_size(preset, profile, image_path=None, aspect='auto'):
    image, image_sha = None, None
    if preset['metadata']['mode'] == 'i2v':
        if not image_path or not Path(image_path).is_file():
            raise FileNotFoundError('Set IMAGE_PATH to the reference in /kaggle/input; image files are not bundled')
        image_sha = hashlib.sha256(Path(image_path).read_bytes()).hexdigest()
        with Image.open(image_path) as source:
            image = ImageOps.exif_transpose(source).convert('RGB')
        if aspect == 'auto':
            ratio = image.width / image.height
            aspect = 'portrait' if ratio < 0.8 else ('square' if ratio <= 1.25 else 'landscape')
    elif image_path:
        raise ValueError('Select an i2v preset when using a reference image')
    aspect = 'landscape' if aspect == 'auto' else aspect
    width, height = SIZES[aspect][profile]
    if image is not None:
        image = ImageOps.pad(image, (width, height), method=Image.Resampling.LANCZOS, color=image.getpixel((0, 0)))
    return image, image_sha, width, height


In [ ]:
def finite(value, name):
    if not bool(torch.isfinite(value).all()):
        raise FloatingPointError(f'{name} contains NaN/Inf; inspect the FP16 path before continuing')


def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def measured(name, operation, reports):
    for i in range(torch.cuda.device_count()):
        torch.cuda.synchronize(i)
        torch.cuda.reset_peak_memory_stats(i)
    start = time.perf_counter()
    result = operation()
    for i in range(torch.cuda.device_count()):
        torch.cuda.synchronize(i)
    # ru_maxrss is a process-lifetime high-water mark, not a per-stage RAM peak.
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    reports[name] = dict(seconds=time.perf_counter()-start,
        process_lifetime_peak_rss_gib=rss/(2**30 if sys.platform == 'darwin' else 2**20),
        gpu=[dict(index=i, allocated_peak_gib=torch.cuda.max_memory_allocated(i)/2**30,
                  reserved_peak_gib=torch.cuda.max_memory_reserved(i)/2**30)
             for i in range(torch.cuda.device_count())])
    print(name, json.dumps(reports[name]), flush=True)
    return result


def model_kwargs(cfg, cache_dir):
    source = cfg['local_model_dir'] or cfg['model_id']
    kwargs = dict(cache_dir=str(cache_dir), local_files_only=bool(cfg['local_model_dir']))
    if not cfg['local_model_dir']:
        kwargs['revision'] = cfg['revision']
    return source, kwargs


def text_key(prompt, cfg):
    return digest(dict(prompt=prompt, revision=cfg['revision'], model=cfg['model_id'],
                       dtype='float16', max_sequence_length=cfg['max_sequence_length'], packages=cfg['packages']))


def compact_conditioning(encoded):
    tokens, pooled, mask = encoded[:3]
    if tokens.shape[0] != 1:
        raise ValueError('This runner encodes one prompt per call')
    # The public pipeline call has no prompt_attention_mask argument. Remove padding
    # after Qwen encoding so cached inputs contain only the valid token sequence.
    if mask is not None:
        tokens = tokens[:, mask[0].bool()]
    finite(tokens, 'Qwen conditioning')
    finite(pooled, 'CLIP conditioning')
    return {'prompt_embeds': tokens.detach().cpu().contiguous().clone(),
            'pooled_prompt_embeds': pooled.detach().cpu().contiguous().clone()}


In [ ]:
NVIDIA_SMI_FIELDS = ('timestamp', 'index', 'utilization.gpu', 'memory.used', 'power.draw')


def summarize_nvidia_smi(path, interval_ms, busy_percent=50):
    # nvidia-smi prints one row per GPU in index order for every polling tick.
    samples, current = [], {}
    for line in Path(path).read_text().splitlines():
        parts = [part.strip() for part in line.split(',')]
        if len(parts) != len(NVIDIA_SMI_FIELDS):
            continue
        try:
            index, util, memory = int(parts[1]), float(parts[2]), float(parts[3])
        except ValueError:
            continue
        if index in current:
            samples.append(current)
            current = {}
        current[index] = (util, memory)
    if current:
        samples.append(current)
    gpus = sorted({index for sample in samples for index in sample})
    samples = [sample for sample in samples if len(sample) == len(gpus)]
    if not samples:
        return dict(available=True, samples=0, interval_ms=interval_ms)
    busy_counts = [sum(sample[i][0] >= busy_percent for i in gpus) for sample in samples]
    share = lambda count: sum(c == count for c in busy_counts)/len(samples)
    return dict(available=True, samples=len(samples), interval_ms=interval_ms, busy_percent=busy_percent,
        gpu=[dict(index=i,
                  mean_util_percent=sum(s[i][0] for s in samples)/len(samples),
                  max_util_percent=max(s[i][0] for s in samples),
                  busy_fraction=sum(s[i][0] >= busy_percent for s in samples)/len(samples),
                  max_memory_used_mib=max(s[i][1] for s in samples)) for i in gpus],
        all_busy_fraction=share(len(gpus)), one_busy_fraction=share(1), none_busy_fraction=share(0))


class NvidiaSmiTrace:
    """Poll per-GPU utilization in a child nvidia-smi process while the wrapped block runs."""

    def __init__(self, path, interval_ms=200):
        self.path, self.interval_ms = Path(path), interval_ms
        self.process, self.summary = None, dict(available=False)

    def __enter__(self):
        self.file = self.path.open('w')
        try:
            self.process = subprocess.Popen(['nvidia-smi', '--query-gpu='+','.join(NVIDIA_SMI_FIELDS),
                '--format=csv,noheader,nounits', '-lms', str(self.interval_ms)],
                stdout=self.file, stderr=subprocess.DEVNULL)
        except FileNotFoundError:
            print('nvidia-smi not found; GPU utilization is not traced', flush=True)
        return self

    def __exit__(self, *exc):
        if self.process is not None:
            self.process.terminate()
            self.process.wait(timeout=10)
        self.file.close()
        if self.process is not None:
            self.summary = summarize_nvidia_smi(self.path, self.interval_ms)
        return False


def plot_nvidia_smi(path, output_path, interval_ms=200):
    import matplotlib.pyplot as plt
    series = {}
    for line in Path(path).read_text().splitlines():
        parts = [part.strip() for part in line.split(',')]
        try:
            series.setdefault(int(parts[1]), []).append(float(parts[2]))
        except (IndexError, ValueError):
            continue
    figure, axis = plt.subplots(figsize=(10, 3))
    for index, values in sorted(series.items()):
        axis.plot([k*interval_ms/1000 for k in range(len(values))], values, label=f'cuda:{index}')
    axis.set(xlabel='seconds since denoising start', ylabel='GPU utilization, %', ylim=(0, 105))
    axis.legend(loc='upper right')
    figure.tight_layout()
    figure.savefig(output_path, dpi=120)
    return figure


In [ ]:
@torch.inference_mode()
def encode_texts(prompts, cfg):
    directory = Path(cfg['results'])/'conditioning'
    directory.mkdir(parents=True, exist_ok=True)
    paths = {name: directory/(text_key(prompt, cfg)+'.safetensors') for name,prompt in prompts.items()}
    missing = {name: prompt for name,prompt in prompts.items() if not paths[name].is_file()}
    if not missing:
        print('Reused saved conditioning for Smoke and Baseline')
        return paths
    if not cfg['local_model_dir'] and shutil.disk_usage(cfg['scratch']).free < 18*2**30:
        raise OSError('Text encoding needs 18 GiB free scratch; attach a full Diffusers snapshot as Input or free scratch')
    # Only this temporary directory owns downloaded text weights. Mounted Input is read-only.
    with tempfile.TemporaryDirectory(prefix='text-weights-', dir=cfg['scratch']) as cache:
        source, kwargs = model_kwargs(cfg, cache)
        device = torch.device('cuda:1')
        qwen = Qwen2_5_VLForConditionalGeneration.from_pretrained(source, subfolder='text_encoder',
            dtype=torch.float16, device_map='auto', max_memory={1: '11GiB', 'cpu': '18GiB'},
            low_cpu_mem_usage=True, attn_implementation='sdpa', **kwargs)
        clip = CLIPTextModel.from_pretrained(source, subfolder='text_encoder_2',
                                            dtype=torch.float32, **kwargs).to(device)
        pipe = Kandinsky6TI2VAPipeline(transformer=None, vae=None, audio_vae=None, vocoder=None,
            text_encoder=qwen, text_encoder_2=clip,
            tokenizer=Qwen2_5_VLProcessor.from_pretrained(source, subfolder='tokenizer', **kwargs),
            tokenizer_2=CLIPTokenizer.from_pretrained(source, subfolder='tokenizer_2', **kwargs),
            scheduler=PiflowScheduler.from_pretrained(source, subfolder='scheduler', **kwargs))
        try:
            print('Qwen device map:', qwen.hf_device_map, flush=True)
            for name,prompt in missing.items():
                encoded = pipe.encode_prompt(prompt, do_classifier_free_guidance=False,
                    device=device, dtype=torch.float16, max_sequence_length=cfg['max_sequence_length'])
                tensors = compact_conditioning(encoded)
                save_file(tensors, str(paths[name]), metadata={'key': text_key(prompt,cfg)})
                print('Encoded', name, list(tensors['prompt_embeds'].shape), flush=True)
                del encoded, tensors
        finally:
            del pipe, qwen, clip
            release_memory()
    return paths


In [ ]:
DIT_PLACEMENTS = ('balanced', 'cuda:0', 'cuda:1')


def dit_load_options(placement):
    # 'balanced' splits blocks for memory; a single device keeps the whole DiT on one T4.
    if placement == 'balanced':
        return dict(device_map='balanced', max_memory={0:'10GiB',1:'10GiB'}), {0,1}
    if placement in DIT_PLACEMENTS:
        return dict(device_map=placement), {int(placement.split(':')[1])}
    raise ValueError(f'DIT_PLACEMENT must be one of {DIT_PLACEMENTS}, got {placement!r}')


def mapped_gpu_indices(device_map):
    devices = set()
    for value in device_map.values():
        if isinstance(value, int):
            devices.add(value)
        else:
            value = str(value)
            if value.startswith('cuda:'):
                devices.add(int(value.split(':')[1]))
    return devices


def load_sampling_pipeline(cfg, has_image):
    source, kwargs = model_kwargs(cfg, cfg['media_cache'])
    placement = cfg['dit_placement']
    options, expected = dit_load_options(placement)
    transformer = Kandinsky6Transformer3DModel.from_pretrained(source, subfolder='transformer',
        torch_dtype=torch.float16, low_cpu_mem_usage=True, **options, **kwargs)
    if transformer.config.model_dim != 1792 or transformer.config.num_visual_blocks != 32:
        raise ValueError('Expected the Lite architecture, not Pro')
    # A single-device load may skip accelerate dispatch and leave no hf_device_map.
    device_map = getattr(transformer, 'hf_device_map', None) or {'': str(transformer.device)}
    if mapped_gpu_indices(device_map) != expected or any(str(v) in ('cpu','disk') for v in device_map.values()):
        raise RuntimeError(f'Expected DiT on GPU(s) {sorted(expected)}, no CPU/disk: {device_map}')
    print('DiT device map:', json.dumps(device_map, default=str), flush=True)
    # Encoder overrides prevent the pipeline from downloading Qwen a second time.
    pipe = Kandinsky6TI2VAPipeline.from_pretrained(source, transformer=transformer,
        text_encoder=None, text_encoder_2=None, tokenizer=None, tokenizer_2=None,
        torch_dtype={'default':torch.float32}, **kwargs)
    if not isinstance(pipe.scheduler, PiflowScheduler) or pipe.scheduler.config.n_grid != 10:
        raise ValueError('Expected the Lite distilled PiFlow scheduler with n_grid=10')
    pipe.vae.enable_tiling()
    if has_image:
        # One reference-frame encode next to the first DiT block; DiT blocks keep their device map.
        pipe.vae.to('cuda:0' if placement == 'balanced' else placement)
    # Do not call pipe.to(...) or enable_model_cpu_offload() over a sharded transformer.
    return pipe, device_map


In [ ]:
@torch.inference_mode()
def sample_joint(cfg, prompt, negative, conditioning_path, image, latent_path, trace_path=None):
    if latent_path.exists():
        with safe_open(str(latent_path), framework='pt') as f:
            if f.metadata().get('job_key') != cfg['job_key']:
                raise ValueError('Cached latent belongs to a different configuration/reference')
        print('Reusing joint latents; no Qwen/DiT rerun')
        return {'cache_reused':True}
    conditioning = load_file(str(conditioning_path))
    with safe_open(str(conditioning_path), framework='pt') as f:
        if f.metadata().get('key') != text_key(prompt,cfg):
            raise ValueError('Stale text conditioning')
    pipe, device_map = load_sampling_pipeline(cfg, image is not None)
    calls = [0]
    def count_call(module, inputs):
        calls[0] += 1
    def progress(pipeline, step, timestep, tensors):
        finite(tensors['latents'], f'video step {step}')
        print(f'DiT {step+1}/10, timestep={float(timestep):.2f}', flush=True)
        return tensors
    hook = pipe.transformer.register_forward_pre_hook(count_call)
    # The trace covers only the denoising call, not weight loading or latent saving.
    trace = NvidiaSmiTrace(trace_path) if trace_path else None
    gpus = sorted(mapped_gpu_indices(device_map))
    try:
        with trace or contextlib.nullcontext():
            for i in gpus:
                torch.cuda.synchronize(i)
            start = time.perf_counter()
            output = pipe(image=image, negative_prompt=negative,
                height=cfg['height'], width=cfg['width'], num_frames=cfg['num_frames'], frame_rate=24,
                num_inference_steps=10, guidance_scale=1.0, sample_audio=True, expand_prompts=False,
                generator=torch.Generator(device='cpu').manual_seed(cfg['seed']),
                output_type='latent', callback_on_step_end=progress, **conditioning)
            for i in gpus:
                torch.cuda.synchronize(i)
            denoise_seconds = time.perf_counter()-start
        finite(output.frames, 'sampled video'); finite(output.audio, 'sampled audio')
        if calls[0] != 10:
            raise AssertionError(f'Expected 10 measured DiT calls, got {calls[0]}')
        save_file({'video':output.frames.detach().cpu().contiguous(),
                   'audio':output.audio.detach().cpu().contiguous()}, str(latent_path),
                  metadata={'job_key':cfg['job_key']})
        result = dict(dit_nfe=calls[0], dit_placement=cfg['dit_placement'], device_map=device_map,
                      denoise_seconds=denoise_seconds, cache_reused=False,
                      video_shape=list(output.frames.shape), audio_shape=list(output.audio.shape))
        if trace is not None:
            result['nvidia_smi'] = trace.summary
        del output
        return result
    finally:
        hook.remove()
        del pipe
        release_memory()


def job_key_for(cfg, prompt, image_sha):
    # Placement is part of the key, so a new placement re-runs denoising instead of reusing latents.
    return digest(dict(model=cfg['model_id'], revision=cfg['revision'], packages=cfg['packages'], prompt=prompt,
        seed=cfg['seed'], width=cfg['width'], height=cfg['height'], num_frames=cfg['num_frames'], image=image_sha,
        steps=10, cfg=1.0, dtype='float16',
        placement='balanced-two-gpu' if cfg['dit_placement'] == 'balanced' else cfg['dit_placement']))


def measure_placements(base_cfg, preset, preset_name, combos, conditioning, image_path=None, aspect='auto'):
    # Denoise-only runs for (profile, placement) pairs; an out-of-memory error is recorded as a result.
    rows = []
    for profile, placement in combos:
        prompt, negative = compile_prompt(preset, profile)
        image, image_sha, width, height = reference_and_size(preset, profile, image_path, aspect)
        cfg = dict(base_cfg, profile=profile, dit_placement=placement, width=width, height=height, **PROFILES[profile])
        cfg['job_key'] = job_key_for(cfg, prompt, image_sha)
        run_dir = Path(cfg['results'])/(preset_name+'-'+profile+'-'+cfg['job_key'][:12])
        run_dir.mkdir(exist_ok=True)
        (run_dir/'config.json').write_text(json.dumps(dict(cfg, prompt=prompt, negative_prompt=negative,
            reference_sha256=image_sha), indent=2))
        trace_path = run_dir/('nvidia-smi-denoise-'+str(time.time_ns())+'.csv')
        reports = {}
        print(f'== Measure {profile} / {placement}', flush=True)
        try:
            sample = measured('sample', lambda: sample_joint(cfg, prompt, negative, conditioning[profile], image,
                run_dir/'joint-latents.safetensors', trace_path), reports)
            sample = dict(sample, stage=reports['sample'])
        except torch.cuda.OutOfMemoryError as error:
            sample = dict(oom=True, dit_placement=placement, error=str(error).splitlines()[0][:300])
            print('Out of memory:', sample['error'], flush=True)
        release_memory()
        (run_dir/'sample.json').write_text(json.dumps(sample, indent=2, default=str))
        rows.append(dict(sample, run=run_dir.name))
    return rows


def compare_placements(results_dir):
    # One row per run directory whose sampling actually ran (not a reused latent cache).
    rows = []
    for sample_path in sorted(Path(results_dir).glob('*/sample.json')):
        sample = json.loads(sample_path.read_text())
        config = json.loads((sample_path.parent/'config.json').read_text())
        size = f"{config['width']}x{config['height']}x{config['num_frames']}"
        if sample.get('oom'):
            rows.append(dict(run=sample_path.parent.name, placement=sample['dit_placement'], size=size,
                             oom=True, error=sample['error']))
            continue
        if sample.get('cache_reused') or 'denoise_seconds' not in sample:
            continue
        trace = sample.get('nvidia_smi', {})
        stage = sample.get('stage', {})
        rows.append(dict(run=sample_path.parent.name, placement=sample['dit_placement'], size=size,
            denoise_s=round(sample['denoise_seconds'], 2),
            peak_alloc_gib=[round(g['allocated_peak_gib'], 2) for g in stage.get('gpu', [])],
            all_busy=trace.get('all_busy_fraction'), one_busy=trace.get('one_busy_fraction')))
    return rows


In [ ]:
@torch.inference_mode()
def decode_video(cfg, latent_path):
    source, kwargs = model_kwargs(cfg, cfg['media_cache'])
    vae = AutoencoderKLHunyuanVideo.from_pretrained(source, subfolder='vae', torch_dtype=torch.float32, **kwargs).to('cuda:1')
    vae.enable_tiling(tile_sample_min_height=256, tile_sample_min_width=256,
        tile_sample_min_num_frames=16, tile_sample_stride_height=192,
        tile_sample_stride_width=192, tile_sample_stride_num_frames=12)
    try:
        z = load_file(str(latent_path))['video'].to('cuda:1',dtype=vae.dtype)
        decoded = vae.decode(z/vae.config.scaling_factor, return_dict=False)[0]
        finite(decoded, 'decoded video')
        frames = VideoProcessor(vae_scale_factor=8).postprocess_video(decoded, output_type='np')[0]
        if tuple(frames.shape) != (cfg['num_frames'], cfg['height'], cfg['width'],3):
            raise ValueError(f'Unexpected decoded frame shape: {frames.shape}')
        return frames[:cfg['export_frames']].copy()
    finally:
        del vae
        release_memory()


@torch.inference_mode()
def decode_audio(cfg, latent_path):
    source, kwargs = model_kwargs(cfg, cfg['media_cache'])
    vae = MMAudioVAE.from_pretrained(source, subfolder='audio_vae', torch_dtype=torch.float32, **kwargs).to('cuda:0')
    vocoder = MMAudioVocoder.from_pretrained(source, subfolder='vocoder', torch_dtype=torch.float32, **kwargs).to('cuda:0')
    try:
        z = load_file(str(latent_path))['audio'].to('cuda:0',dtype=vae.dtype)
        mel = vae.decode(z/vae.config.scaling_factor, return_dict=False)[0]
        waveform = vocoder(mel.to(vocoder.dtype), return_dict=False)[0][0,0].float().cpu()[None]
        finite(waveform, 'waveform')
        rate = vae.config.sample_rate
        samples = round(cfg['export_frames']/24*rate)
        if waveform.shape[-1] < samples:
            raise ValueError('Decoded audio is shorter than the requested export')
        return waveform[:,:samples].contiguous(), rate
    finally:
        del vae, vocoder
        release_memory()


In [ ]:
def export_clip(frames, waveform, rate, cfg, path):
    import av
    if path.exists():
        raise FileExistsError(f'Choose a new output name to preserve the previous clip: {path}')
    encode_video(frames, fps=24, output_path=str(path), audio=waveform, audio_sample_rate=rate)
    with av.open(str(path)) as container:
        video = container.streams.video[0]
        if len(container.streams.audio) != 1:
            raise ValueError('Expected one audio stream')
        if (video.width,video.height)!=(cfg['width'],cfg['height']):
            raise ValueError('Wrong MP4 geometry')
        fps=float(video.average_rate)
        count=sum(1 for _ in container.decode(video=0))
    with av.open(str(path)) as container:
        audio_samples=sum(f.samples for f in container.decode(audio=0))
    if count!=cfg['export_frames'] or fps!=24 or abs(audio_samples/rate-count/fps)>0.15:
        raise ValueError('MP4 frame/FPS/audio-duration verification failed')
    return dict(mp4=str(path), exported_frames=count, fps=fps, seconds=count/fps,
                audio_sample_rate=rate, decoded_audio_samples=audio_samples)


In [ ]:
def soundtrack_audio(generated, rate, music_path=None, offset_seconds=0.0, mode='generated', gain=0.65):
    if mode == 'generated':
        return generated, {'source':'generated'}
    if mode not in ('music','mix') or not music_path or offset_seconds < 0 or not 0 <= gain <= 1:
        raise ValueError('Set a local MUSIC_PATH, nonnegative offset, mode music/mix and gain in [0,1]')
    import av
    wanted = generated.shape[-1]
    skip = round(offset_seconds*rate)
    chunks, remaining = [], wanted
    with av.open(str(music_path)) as container:
        resampler = av.AudioResampler(format='fltp', layout='mono', rate=rate)
        for frame in container.decode(audio=0):
            for converted in resampler.resample(frame):
                values = torch.from_numpy(converted.to_ndarray().copy()).float()
                dropped = min(skip,values.shape[-1]);skip-=dropped
                values=values[:,dropped:]
                if values.shape[-1]:
                    part=values[:,:remaining];chunks.append(part);remaining-=part.shape[-1]
                if remaining == 0:break
            if remaining == 0:break
    if remaining:
        raise ValueError('Music file is shorter than offset plus clip duration')
    music=torch.cat(chunks,dim=-1)*gain
    mixed=music if mode=='music' else generated*0.65+music
    finite(mixed,'soundtrack')
    peak=float(mixed.abs().max())
    if peak>0.98:mixed=mixed*(0.98/peak)
    # Short fades avoid clicks without changing the requested number of samples.
    n=min(round(rate*0.01),wanted//2)
    mixed[:,:n]*=torch.linspace(0,1,n)
    mixed[:,-n:]*=torch.linspace(1,0,n)
    return mixed.contiguous(),{'source':mode,'filename':Path(music_path).name,
                              'offset_seconds':offset_seconds,'gain':gain,'mono':True}


## 4. Конфигурация, reference и prompt


In [ ]:
preset=PRESETS[PRESET_NAME]
PROMPTS={name:compile_prompt(preset,name)[0] for name in PROFILES}
PROMPT,NEGATIVE=compile_prompt(preset,PROFILE)
REFERENCE,REFERENCE_SHA,WIDTH,HEIGHT=reference_and_size(preset,PROFILE,IMAGE_PATH,ASPECT)
CFG=dict(model_id=MODEL_ID,revision=MODEL_REVISION,local_model_dir=LOCAL_MODEL_DIR,
    scratch=str(SCRATCH),results=str(RESULTS),media_cache=str(SCRATCH/'media-cache'),
    packages=pins,max_sequence_length=1024,seed=SEED,profile=PROFILE,dit_placement=DIT_PLACEMENT,
    width=WIDTH,height=HEIGHT,**PROFILES[PROFILE])
CFG['job_key']=job_key_for(CFG,PROMPT,REFERENCE_SHA)
RUN_DIR=RESULTS/(PRESET_NAME+'-'+PROFILE+'-'+CFG['job_key'][:12])
RUN_DIR.mkdir(exist_ok=True)
LATENTS=RUN_DIR/'joint-latents.safetensors'
REPORTS={}
(RUN_DIR/'config.json').write_text(json.dumps(dict(CFG,prompt=PROMPT,negative_prompt=NEGATIVE,
    reference_sha256=REFERENCE_SHA),indent=2))
print(json.dumps(CFG,indent=2));print(PROMPT)
print('Qwen text phase needs about 18 GiB of free scratch unless weights come from Input.')
if REFERENCE is not None:
    from IPython.display import display
    display(REFERENCE)
if AUDIO_MODE in ('music','mix'):
    # Validate the uploaded track and preview the chosen excerpt before any model download.
    if not MUSIC_PATH or not Path(MUSIC_PATH).is_file():
        raise FileNotFoundError('Upload your AAC/audio file to Kaggle Input and set its actual MUSIC_PATH before generation.')
    MUSIC_PREVIEW,MUSIC_SOURCE=soundtrack_audio(torch.zeros(1,CFG['seconds']*44100),44100,
        MUSIC_PATH,MUSIC_OFFSET_SECONDS,'music',MUSIC_GAIN)
    (RUN_DIR/'music-source.json').write_text(json.dumps(MUSIC_SOURCE,indent=2))
    from IPython.display import Audio,display
    display(Audio(MUSIC_PREVIEW.numpy(),rate=44100,normalize=False))
    print('Soundtrack checked; preview uses the selected offset and gain.')


## 5. Текст и joint sampling

Оба prompt-cache сохраняются в Output. Скачать text-веса потребуется снова только при новом/изменённом preset; private text cache удаляется после encoding. Самплирование сохраняет joint latents и фоновую трассировку `nvidia-smi`. При ошибке decoder повторяйте раздел 6: Qwen и DiT заново не нужны.


In [ ]:
CONDITIONING=measured('text',lambda:encode_texts(PROMPTS,CFG),REPORTS)


In [ ]:
GPU_TRACE=RUN_DIR/('nvidia-smi-denoise-'+str(time.time_ns())+'.csv')
SAMPLE=measured('sample',lambda:sample_joint(CFG,PROMPT,NEGATIVE,CONDITIONING[PROFILE],REFERENCE,LATENTS,GPU_TRACE),REPORTS)
(RUN_DIR/'sample.json').write_text(json.dumps(dict(SAMPLE,stage=REPORTS['sample']),indent=2,default=str))
print(SAMPLE)


### Загрузка GPU во время denoising

`nvidia-smi` опрашивается каждые 200 ms в фоне только на время вызова pipeline (без загрузки весов). Summary попадает в `sample.json` и `metrics.json`, CSV и PNG — в Output. `all_busy_fraction` — доля отсчётов, где обе GPU загружены ≥ 50 %: при `device_map='balanced'` и batch 1 ожидается, что GPU работают поочерёдно. Это `utilization.gpu` драйвера (доля времени с активным kernel), не FLOPs. При повторном использовании сохранённых latents трассировки нет.


In [ ]:
if SAMPLE.get('nvidia_smi',{}).get('samples'):
    from IPython.display import display
    import matplotlib.pyplot as plt
    display(plot_nvidia_smi(GPU_TRACE,GPU_TRACE.with_suffix('.png')))
    plt.close('all')
    print(json.dumps(SAMPLE['nvidia_smi'],indent=2))
else:
    print('No nvidia-smi samples:',SAMPLE.get('nvidia_smi','latents reused, denoising skipped'))


### Дополнительные замеры размещений (необязательно)

Если `MEASURE_PLACEMENTS` не пуст, ячейка по очереди выполняет только denoising для каждой пары `(profile, placement)` того же preset и seed, с трассой `nvidia-smi`. Декодирование и MP4 для них не делаются. Нехватка памяти (OOM) записывается в `sample.json` как результат и не останавливает notebook. Подходит для «Save & Run All»: все варианты попадают в одну таблицу ниже.


In [ ]:
if MEASURE_PLACEMENTS:
    MEASUREMENTS=measure_placements(CFG,preset,PRESET_NAME,MEASURE_PLACEMENTS,CONDITIONING,IMAGE_PATH,ASPECT)
else:
    print('MEASURE_PLACEMENTS is empty; no extra measurements.')


### Сравнение размещений DiT

Запустите разделы 4–5 с `DIT_PLACEMENT='balanced'`, затем с `'cuda:0'` при том же preset, profile и seed: placement входит в `job_key`, поэтому второй прогон заново выполняет denoising, а не берёт latents из кэша. `denoise_s` измеряет только вызов pipeline после синхронизации CUDA; `peak_alloc_gib` — пик PyTorch allocator за всю стадию sample, включая загрузку. Один прогон на вариант — ориентир, не статистика.


In [ ]:
for row in compare_placements(RESULTS):
    print(json.dumps(row))


## 6. Video / audio decode из сохранённых latents


In [ ]:
import numpy as np
FRAMES=measured('video',lambda:decode_video(CFG,LATENTS),REPORTS)
np.save(RUN_DIR/'frames.npy',FRAMES,allow_pickle=False)
release_memory()


In [ ]:
(WAVEFORM,SAMPLE_RATE)=measured('audio',lambda:decode_audio(CFG,LATENTS),REPORTS)
np.save(RUN_DIR/'waveform.npy',WAVEFORM.numpy(),allow_pickle=False)
release_memory()


## 7. MP4 и измерения


In [ ]:
FRAMES=np.load(RUN_DIR/'frames.npy',allow_pickle=False)
WAVEFORM=torch.from_numpy(np.load(RUN_DIR/'waveform.npy',allow_pickle=False))
WAVEFORM,SOUNDTRACK=soundtrack_audio(WAVEFORM,SAMPLE_RATE,MUSIC_PATH,MUSIC_OFFSET_SECONDS,AUDIO_MODE,MUSIC_GAIN)
# A fresh name preserves the previous export when this cell is repeated.
OUTPUT=RUN_DIR/('clip-'+str(time.time_ns())+'.mp4')
EXPORTED=measured('export',lambda:export_clip(FRAMES,WAVEFORM,SAMPLE_RATE,CFG,OUTPUT),REPORTS)
(RUN_DIR/'metrics.json').write_text(json.dumps(dict(stages=REPORTS,sample=SAMPLE,export=EXPORTED,soundtrack=SOUNDTRACK),indent=2,default=str))
from IPython.display import Video,display
display(Video(str(OUTPUT),embed=True))
print(json.dumps(EXPORTED,indent=2))


Сохраните MP4, `config.json`, `metrics.json` и `joint-latents.safetensors` через Kaggle Save Version / Output. Временная среда и media cache в `/kaggle/temp` не являются архивом. Source-level/CPU проверки находятся в GitHub; успешный Quick Save на Kaggle не доказывает pretrained inference.
